# 02: Causal Masking (Global + Sliding Window)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/gemma_from_scratch/blob/main/workshop/02_causal_masking.ipynb)

[← Previous: 01 The Math of Attention](01_the_math_of_attention.ipynb) | [Next: 03 Grouped Query Attention →](03_grouped_query_attention.ipynb)


**Estimated Time: 10 minutes**

In a decoder-only model like Gemma 3, when we predict the next token, we must ensure the model only uses information from the current and previous tokens. If the model can "see" the answer during training, it won't learn anything!

---
## Learning Objectives
1. Understand the concept of Causal Masking.
2. Implement a triangular mask using `torch.triu`.
3. Learn about Gemma 3's **5:1 local/global layer interleaving** with sliding window.

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

seq_len = 12
sliding_window = 4  # Gemma 3 uses sliding_window=1024

---
## 🔒 The Global Causal Mask: Safeguarding Autoregressive Generation

Language models generate text one token at a time. To train them efficiently in parallel, we feed the entire sequence at once but use a Causal Mask to ensure a token at position $i$ can only compute attention scores using Keys from positions $j \le i$.

### 🧮 Causal Mask Construction:
1. We initialize a boolean matrix of shape `(seq_len, seq_len)` containing all `True` values (`1`).
2. We extract the upper triangle of this matrix using `torch.triu(..., diagonal=1)`. This marks future positions as `True` (masked), and past positions as `False` (visible).

   $$\text{Mask}_{ij} = \begin{cases} \text{True} & \text{if } j > i \\ \text{False} & \text{if } j \le i \end{cases}$$

3. Inside the attention layer, these `True` positions will be filled with $-\infty$. Because $\exp(-\infty) = 0$, applying the softmax operation completely zeroes out attention to future tokens, preventing the model from "looking ahead" during training!

In PyTorch, this is elegantly handled using an upper-triangular matrix function:
```python
mask_global = torch.triu(ones, diagonal=1)
```

This creates a boolean grid where everything above the main diagonal is True (meaning "mask this"), and everything below or on the diagonal is False (meaning "allow this").

In [10]:
ones = torch.ones((seq_len, seq_len), dtype=torch.bool)
# Upper triangle (excluding diagonal) = future positions the model MUST NOT see
mask_global = torch.triu(ones, diagonal=1)

plt.figure(figsize=(6, 2.5))
plt.imshow(mask_global.float(), cmap="gray_r")
plt.title("Global Causal Mask (White=Visible, Black=Masked)")
plt.xlabel("Key position (what it looks at)")
plt.ylabel("Query position")
plt.tight_layout()
plt.show()

---
## 🚀 Sliding Window Attention (SWA) Causal Masking

Standard Multi-Head Attention requires $O(N^2)$ memory and compute, making large contexts devastating to GPU VRAM due to the massive Key-Value cache.

Commercial Gemma 3 models must scale up to **128,000 tokens**. Storing key-value pairs for 128K context in Multi-Head Attention consumes massive GPU memory. To solve this, Gemma 3 uses **Sliding Window Attention (SWA)**.

### 📐 Sliding Window Concept:
A token at position $i$ can only see a window of the most recent past tokens (e.g., sliding window $W = 1024$). Tokens that are further back than $W$ steps are masked out:

$$\text{Visible Range for Query } i: \quad [i - W + 1, \quad i]$$

### 🧮 SWA Mask Construction:
1. **Far Past Detection**: We find positions too far in the past using `torch.tril(..., diagonal=-sliding_window)`. This flags positions where $j \le i - W$ as `True`.
2. **Local Mask Assembly**: We combine the standard global causal mask and the far-past mask using a bitwise OR operation (`|`):

   $$\text{Mask}_{local} = \text{Mask}_{global} \cup \text{Mask}_{far\_past}$$

   This creates a diagonal band of visibility, freeing up KV cache during generation!

In [ ]:
# For the sliding window: tokens too far in the past are masked
# With sliding_window=4, query i can see keys j where i-3 <= j <= i.
# Tokens at j <= i - sliding_window are masked out.
ones = torch.ones((seq_len, seq_len), dtype=torch.bool)

# far_past[i, j] = True if j <= i - sliding_window (key is too far back)
far_past = torch.tril(ones, diagonal=-sliding_window)

# Local attention = causal AND recent only
mask_local = mask_global | far_past

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(mask_global.float(), cmap="gray_r")
plt.title("Global Mask (all past)")
plt.xlabel("Keys")
plt.ylabel("Queries")

plt.subplot(1, 2, 2)
plt.imshow(mask_local.float(), cmap="gray_r")
plt.title(f"Local Mask (window={sliding_window})")
plt.xlabel("Keys")
plt.ylabel("Queries")
plt.tight_layout()
plt.show()

> **Note:** black cells are masked.

---
## 🔄 Alternating 5:1 Local/Global Attention Pattern

If every layer in our model used a sliding window of 1024 tokens, information could never flow across long sequences (e.g. token 0 could never propagate information to token 5000). 

To solve this, Gemma 3 uses a **hybrid alternating 5:1 layer pattern**:
- **5 Local Layers**: Layers that employ Sliding Window Attention (SWA) to build fast, local contexts with tiny KV cache demands.
- **1 Global Layer**: A layer that performs full attention over the entire context history, serving as an information bridge.

### 🧮 Pattern Selection Formula:
We determine the attention type of a block based on its index:

$$\text{Attention Type} = \begin{cases} \text{Global} & \text{if } \text{layer\_idx} \pmod 6 == 5 \\ \text{Local} & \text{otherwise} \end{cases}$$

In our 8-layer pedagogical model, only **Layer 5** (the 6th layer, 0-indexed) is Global, while all other layers are Local. This achieves the perfect balance of localized memory savings and long-range coherence!

In [4]:
def get_layer_mask_type(layer_idx, sliding_window=1024):
    """Determine if layer is global or local.
    Gemma 3 pattern: layers 0-4 are local, layer 5 is global, repeat.
    """
    if layer_idx % 6 == 5:  # Every 6th layer (0-indexed) is global
        return "global"
    return "local"


# Show the pattern for 12 layers
pattern = [(i, get_layer_mask_type(i)) for i in range(12)]
print("Layer pattern (5 local : 1 global):")
for layer, mask_type in pattern:
    bar = "█" * (1024 if mask_type == "local" else 1) + "▌"
    print(
        f"  Layer {layer:2d}: {mask_type:6s} |{bar[:12]}| "
        + ("full context" if mask_type == "global" else f"window={1024}")
    )

print("\nEvery 6th layer is GLOBAL (full attention)")
print("Other 5 layers are LOCAL (sliding window=1024)")

---
## Mask Application inside Attention Calculations

***We apply the causal mask directly to attention scores before the softmax operation to filter out disallowed connections.***

### 🧮 Mathematical Sequence:
1. **Attention Scores**: We calculate raw cosine similarity dot products.
2. **Masking**: Boolean `True` positions in the mask are filled with $-\infty$ using `.masked_fill(..., float("-inf"))`:

   $$\text{Scores}_{ij} \leftarrow \begin{cases} -\infty & \text{if } \text{Mask}_{ij} \text{ is True} \\ \text{Scores}_{ij} & \text{otherwise} \end{cases}$$

3. **Softmax**:

   $$\text{Weights}_{ij} = \frac{e^{\text{Scores}_{ij}}}{\sum_k e^{\text{Scores}_{ik}}}$$

   Since $e^{-\infty} = 0.0$, all masked positions are assigned an exact attention weight of **0.0**, ensuring the model cannot retrieve future token representations!

Here is now the Attention formula:

$$Attention(Q, K, V) = \text{Softmax}\left(\frac{\text{RMSNorm}(Q) \cdot \text{RMSNorm}(K)^T}{\sqrt{d_k}} + M\right) \cdot V$$

Why we Add ($+$) the Mask:

- For Allowed Tokens (The Black Areas): The mask value is $0$.

  $$Score + 0 = Score$$

  The raw dot-product survives perfectly intact.

- For Blocked Tokens (The White/Green Areas): The mask value is $-\infty$.

  $$Score + (-\infty) = -\infty$$

  No matter how high the original dot-product score was, adding negative infinity instantly destroys it.

In [ ]:
# Generating the "Fake" Attention Matrix
scores = torch.randn(seq_len, seq_len)
# Injecting the Mask
masked_scores = scores.masked_fill(mask_global, float("-inf"))

print("Raw Scores (top-left 3x3):\n", scores[:3, :3].round(decimals=3))
print("\nMasked Scores (top-left 3x3):\n", masked_scores[:3, :3].round(decimals=3))
# Note: masked positions become -inf, so softmax gives 0.0

# Applying the Softmax
weights = torch.softmax(masked_scores, dim=-1)
print(
    "\nAttention Weights (zeros in upper triangle):\n",
    weights[:3, :3].round(decimals=5),
)
print("Each row sums to:", weights.sum(dim=-1))

---
## Exercise
Create a function `get_mixed_mask(seq_len, sliding_window)` that returns both global and local masks, plus a function `apply_mask(scores, mask_type, sliding_window)` that applies the right mask based on layer type.

In [6]:
def get_mixed_mask(seq_len, sliding_window=1024):
    """Return both global and local masks for Gemma 3."""
    ones = torch.ones((seq_len, seq_len), dtype=torch.bool)

    # Global mask: upper triangle (future tokens masked)
    # TODO: Implement global mask here!
    mask_global = torch.triu(ones, diagonal=1)

    if mask_global is Ellipsis:
        raise NotImplementedError("Implement Global mask before continuing!")

    # Local mask: also mask keys that are more than sliding_window steps in the past
    far_past = torch.tril(ones, diagonal=-sliding_window)
    mask_local = mask_global | far_past

    return mask_global, mask_local


mask_g, mask_l = get_mixed_mask(seq_len, sliding_window)
assert torch.equal(mask_g, mask_global)
print("✅ get_mixed_mask works correctly!")
print(f"Global mask total masked: {mask_g.sum().item()}")
print(f"Local mask total masked: {mask_l.sum().item()}")

<details>
<summary><b>Click to see solution</b></summary>

```python
def get_mixed_mask(seq_len, sliding_window=1024):
    """Return both global and local masks for Gemma 3."""
    ones = torch.ones((seq_len, seq_len), dtype=torch.bool)

    # Global mask: upper triangle (future tokens masked)
    mask_global = torch.triu(ones, diagonal=1)

    # Local mask: also mask keys that are more than sliding_window steps in the past
    far_past = torch.tril(ones, diagonal=-sliding_window)
    mask_local = mask_global | far_past

    return mask_global, mask_local
```
</details>

---
## Summary

Gemma 3 uses **5 local layers + 1 global layer** pattern:
- **Local layers**: attend to recent window (1024 tokens) + causal → tiny KV cache
- **Global layers**: attend to everything → long-range understanding
- This dramatically reduces inference memory while maintaining quality

[← Previous: 01 The Math of Attention](01_the_math_of_attention.ipynb) | [Next: 03 Grouped Query Attention →](03_grouped_query_attention.ipynb)